In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",None)
%matplotlib inline

In [2]:
df=pd.read_excel("/kaggle/input/buy-or-not-purchase-intent-prediction-challenge/Training Dataset.xlsx")

In [3]:
df.head()

,ID,Start Date,End Date,Estimated Win Rate,Price,Customer Segment 1,Customer Segment 2,Customer Segment 3,Customer Segment 4,Customer Segment 5,Unit Number,Activity 1,Activity 2,Country,City,Product,Competitor,Marketing Source,Division,Manager,Techincal Manager,Customer ID,Result
0,1,2021-03-16,2021-04-20,0.75,3270.83,I,A,AA,CCC,AB,1,EF,IJ,JKL,SME278,VUI150,AAB162,PWH228,PN73,NaN,NaN,426201,0
1,2,2021-03-18,2021-08-16,0.50,3772.74,I,A,AA,CCC,AB,1,EF,IJ,JKL,SME278,WWQ705,AAB162,PWH228,PN73,NaN,NaN,572438,0
2,3,2021-04-01,2021-09-03,0.10,2230.27,I,A,AA,CCC,AB,1,EF,IJ,JKL,SME278,VUI150,AAB162,PWH228,PN73,NaN,NaN,2080929,0
3,4,2021-04-08,2021-07-01,0.50,5000.00,I,A,AA,CCC,AB,1,EF,IJ,JKL,SME278,WWQ705,AAB162,PWH228,PN73,NaN,NaN,959569,0
4,5,2021-03-17,2023-08-24,0.20,51506.23,I,A,AA,CCC,AB,1,EF,IJ,JKL,SME278,OXC337,AAB162,PWH228,EM41,NaN,NaN,229558,0


In [4]:
df.drop(columns=["ID","Customer ID","Manager","Techincal Manager"],axis=1,inplace=True)

In [5]:
df.shape

(177086, 19)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 177086 entries, 0 to 177085
Data columns (total 19 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Start Date          177086 non-null  datetime64[ns]
 1   End Date            177086 non-null  datetime64[ns]
 2   Estimated Win Rate  166527 non-null  float64       
 3   Price               177086 non-null  float64       
 4   Customer Segment 1  177086 non-null  object        
 5   Customer Segment 2  177086 non-null  object        
 6   Customer Segment 3  177086 non-null  object        
 7   Customer Segment 4  177086 non-null  object        
 8   Customer Segment 5  177086 non-null  object        
 9   Unit Number         177086 non-null  int64         
 10  Activity 1          177086 non-null  object        
 11  Activity 2          177086 non-null  object        
 12  Country             177086 non-null  object        
 13  City                177030 no

In [7]:
mean=df["Estimated Win Rate"].mean()
df["Estimated Win Rate"]=df["Estimated Win Rate"].fillna(mean)

mode=df["City"].mode()[0]
df["City"]=df["City"].fillna(mode)


mode=df["Product"].mode()[0]
df["Product"]=df["Product"].fillna(mode)

mode=df["Competitor"].mode()[0]
df["Competitor"]=df["Competitor"].fillna(mode)

mode=df["Marketing Source"].mode()[0]
df["Marketing Source"]=df["Marketing Source"].fillna(mode)

mode=df["Division"].mode()[0]
df["Division"]=df["Division"].fillna(mode)

In [8]:
df.isnull().sum()

Start Date            0
End Date              0
Estimated Win Rate    0
Price                 0
Customer Segment 1    0
Customer Segment 2    0
Customer Segment 3    0
Customer Segment 4    0
Customer Segment 5    0
Unit Number           0
Activity 1            0
Activity 2            0
Country               0
City                  0
Product               0
Competitor            0
Marketing Source      0
Division              0
Result                0
dtype: int64

In [9]:
df['Start Date'] = pd.to_datetime(df['Start Date'])
df['End Date'] = pd.to_datetime(df['End Date'])

In [10]:
df['Start_Year'] = df['Start Date'].dt.year
df['Start_Month'] = df['Start Date'].dt.month
df['Start_Day'] = df['Start Date'].dt.day
df['Start_Weekday'] = df['Start Date'].dt.weekday  # Monday=0

# From End Date
df['End_Year'] = df['End Date'].dt.year
df['End_Month'] = df['End Date'].dt.month
df['End_Day'] = df['End Date'].dt.day
df['End_Weekday'] = df['End Date'].dt.weekday

In [11]:
df.drop(columns=["Start Date","End Date"],axis=1,inplace=True)

In [12]:
df.head()

,Estimated Win Rate,Price,Customer Segment 1,Customer Segment 2,Customer Segment 3,Customer Segment 4,Customer Segment 5,Unit Number,Activity 1,Activity 2,Country,City,Product,Competitor,Marketing Source,Division,Result,Start_Year,Start_Month,Start_Day,Start_Weekday,End_Year,End_Month,End_Day,End_Weekday
0,0.75,3270.83,I,A,AA,CCC,AB,1,EF,IJ,JKL,SME278,VUI150,AAB162,PWH228,PN73,0,2021,3,16,1,2021,4,20,1
1,0.50,3772.74,I,A,AA,CCC,AB,1,EF,IJ,JKL,SME278,WWQ705,AAB162,PWH228,PN73,0,2021,3,18,3,2021,8,16,0
2,0.10,2230.27,I,A,AA,CCC,AB,1,EF,IJ,JKL,SME278,VUI150,AAB162,PWH228,PN73,0,2021,4,1,3,2021,9,3,4
3,0.50,5000.00,I,A,AA,CCC,AB,1,EF,IJ,JKL,SME278,WWQ705,AAB162,PWH228,PN73,0,2021,4,8,3,2021,7,1,3
4,0.20,51506.23,I,A,AA,CCC,AB,1,EF,IJ,JKL,SME278,OXC337,AAB162,PWH228,EM41,0,2021,3,17,2,2023,8,24,3


In [13]:
cat_cols=df.select_dtypes(include=["object"]).columns

mappings = {}

for col in cat_cols:
    codes, uniques = pd.factorize(df[col])
    df[col] = codes
    mappings[col] = dict(enumerate(uniques))

# Show mappings for each column
for col, mapping in mappings.items():
    print(f"{col} mapping: {mapping}\n")

Customer Segment 1 mapping: {0: 'I', 1: 'F', 2: 'G', 3: 'J', 4: 'K', 5: 'H', 6: 'L'}

Customer Segment 2 mapping: {0: 'A', 1: 'C', 2: 'B', 3: 'D', 4: 'E'}

Customer Segment 3 mapping: {0: 'AA', 1: 'BB', 2: 'EE', 3: 'CC', 4: 'DD', 5: 'FF', 6: 'GG'}

Customer Segment 4 mapping: {0: 'CCC', 1: 'BBB', 2: 'AAA', 3: 'DDD'}

Customer Segment 5 mapping: {0: 'AB', 1: 'CD'}

Activity 1 mapping: {0: 'EF', 1: 'GH'}

Activity 2 mapping: {0: 'IJ', 1: 'KL'}

Country mapping: {0: 'JKL', 1: 'VWX', 2: 'DEF', 3: 'PQR', 4: 'STU', 5: 'MNO', 6: 'ABC', 7: 'GHI'}

City mapping: {0: 'SME278', 1: 'YGS695', 2: 'IMF435', 3: 'FUH372', 4: 'SAO455', 5: 'HJR277', 6: 'QSW239', 7: 'JMJ759', 8: 'IQH555', 9: 'VFR844', 10: 'IKB920', 11: 'XJJ260', 12: 'EOH323', 13: 'QGD703', 14: 'EAS271', 15: 'LPJ621', 16: 'VME946', 17: 'ISH651', 18: 'UCX355', 19: 'ISG124', 20: 'DIF777', 21: 'LJK362', 22: 'LNO392', 23: 'OHG455', 24: 'GHK468'}

Product mapping: {0: 'VUI150', 1: 'WWQ705', 2: 'OXC337', 3: 'BNX965', 4: 'OJK516', 5: 'RWN676', 6:

In [14]:
df["Result"].value_counts()

Result
0    101421
1     75665
Name: count, dtype: int64

In [15]:
df.head()

,Estimated Win Rate,Price,Customer Segment 1,Customer Segment 2,Customer Segment 3,Customer Segment 4,Customer Segment 5,Unit Number,Activity 1,Activity 2,Country,City,Product,Competitor,Marketing Source,Division,Result,Start_Year,Start_Month,Start_Day,Start_Weekday,End_Year,End_Month,End_Day,End_Weekday
0,0.75,3270.83,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,2021,3,16,1,2021,4,20,1
1,0.50,3772.74,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,2021,3,18,3,2021,8,16,0
2,0.10,2230.27,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,2021,4,1,3,2021,9,3,4
3,0.50,5000.00,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,2021,4,8,3,2021,7,1,3
4,0.20,51506.23,0,0,0,0,0,1,0,0,0,0,2,0,0,1,0,2021,3,17,2,2023,8,24,3


In [16]:
X = df.drop(columns=['Result'])
y = df['Result']

In [17]:
from sklearn.model_selection import train_test_split

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [18]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(iterations=1000,depth=6,learning_rate=0.1,loss_function='Logloss',random_seed=42,verbose=100)
model.fit(X_train,y_train)

0:	learn: 0.4818861	total: 85.6ms	remaining: 1m 25s
100:	learn: 0.0353990	total: 2.36s	remaining: 21s
200:	learn: 0.0313326	total: 4.62s	remaining: 18.4s
300:	learn: 0.0289217	total: 6.78s	remaining: 15.7s
400:	learn: 0.0273296	total: 8.88s	remaining: 13.3s
500:	learn: 0.0259781	total: 11s	remaining: 11s
600:	learn: 0.0245372	total: 13.1s	remaining: 8.72s
700:	learn: 0.0234074	total: 15.2s	remaining: 6.49s
800:	learn: 0.0220164	total: 17.4s	remaining: 4.32s
900:	learn: 0.0210207	total: 19.4s	remaining: 2.13s
999:	learn: 0.0199110	total: 21.5s	remaining: 0us


In [19]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

# Predict labels
y_pred = model.predict(X_valid)

# Predict probabilities for AUC
y_pred_proba = model.predict_proba(X_valid)[:, 1]

# Metrics
accuracy = accuracy_score(y_valid, y_pred)
precision = precision_score(y_valid, y_pred)
recall = recall_score(y_valid, y_pred)
f1 = f1_score(y_valid, y_pred)
auc = roc_auc_score(y_valid, y_pred_proba)
cm = confusion_matrix(y_valid, y_pred)
report = classification_report(y_valid, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("ROC AUC:", auc)
print("Confusion Matrix:\n", cm)
print("Classification Report:\n", report)


Accuracy: 0.9909650460217968
Precision: 0.9867253729381613
Recall: 0.9922024714200753
F1 Score: 0.9894563426688633
ROC AUC: 0.9978231759575001
Confusion Matrix:
 [[20083   202]
 [  118 15015]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99     20285
           1       0.99      0.99      0.99     15133

    accuracy                           0.99     35418
   macro avg       0.99      0.99      0.99     35418
weighted avg       0.99      0.99      0.99     35418



In [20]:
test_df=pd.read_excel("/kaggle/input/buy-or-not-purchase-intent-prediction-challenge/Testing Dateset.xlsx")

In [21]:
test_df = test_df.reset_index(drop=True)

In [22]:
test_df.head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,ID,Start Date,End Date,Estimated Win Rate,Price,Customer Segment 1,Customer Segment 2,Customer Segment 3,Customer Segment 4,Customer Segment 5,Unit Number,Activity 1,Activity 2,Country,City,Product,Competitor,Marketing Source,Division,Manager,Techincal Manager,Customer ID,Result
0,177087,2018-02-11,2018-09-04,1.00,3000.000000,I,A,AA,CCC,AB,2.0,EF,IJ,GHI,LNO392,VUI150,NaN,JWP393,EM41,NaN,NaN,0017Q00001G4YkIQAV,NaN
1,177088,2018-11-14,2019-03-05,0.75,250000.000000,I,A,AA,CCC,AB,1.0,EF,IJ,GHI,LNO392,IEJ418,NaN,JWP393,QO91,NaN,NaN,0017Q00001G4YheQAF,NaN
2,177089,2024-05-16,2024-08-26,0.00,139677.231907,I,D,AA,CCC,AB,1.0,EF,IJ,PQR,FUH372,AHU981,UTG383,NaN,UW72,NaN,NaN,0017Q00001G4UAOQA3,NaN
3,177090,2023-02-09,2023-08-21,0.00,5194.576510,I,D,AA,CCC,AB,1.0,EF,IJ,MNO,DIF777,ASH122,SVE211,KAD664,EM41,NaN,NaN,0017Q00001OcHuHQAV,NaN
4,177091,2025-02-11,2025-12-31,0.00,132065.504490,I,A,AA,CCC,AB,1.0,EF,IJ,PQR,FUH372,AHU981,UTG383,RMD495,UW72,NaN,NaN,0017Q00001G4ZolQAF,NaN


In [23]:
Id = test_df.ID

In [24]:
test_df.drop(columns=["Manager","Techincal Manager","Customer ID","Result","ID"],axis=1,inplace=True)

In [25]:
test_df.isnull().sum()

Start Date                0
End Date                  0
Estimated Win Rate        0
Price                     0
Customer Segment 1        0
Customer Segment 2        0
Customer Segment 3        0
Customer Segment 4        0
Customer Segment 5        0
Unit Number            4291
Activity 1                0
Activity 2                0
Country                   0
City                   2998
Product                5357
Competitor            37515
Marketing Source       1041
Division                 16
dtype: int64

In [26]:
mode=test_df["City"].mode()[0]
test_df["City"]=test_df["City"].fillna(mode)


mode=test_df["Product"].mode()[0]
test_df["Product"]=test_df["Product"].fillna(mode)

mode=test_df["Competitor"].mode()[0]
test_df["Competitor"]=test_df["Competitor"].fillna(mode)

mode=test_df["Marketing Source"].mode()[0]
test_df["Marketing Source"]=test_df["Marketing Source"].fillna(mode)

mode=test_df["Division"].mode()[0]
test_df["Division"]=test_df["Division"].fillna(mode)


mode=test_df["Unit Number"].mode()[0]
test_df["Unit Number"]=test_df["Unit Number"].fillna(mode)

In [27]:
test_df['Start Date'] = pd.to_datetime(test_df['Start Date'])
test_df['End Date'] = pd.to_datetime(test_df['End Date'])

In [28]:
# From Start Date
test_df['Start_Year'] = test_df['Start Date'].dt.year
test_df['Start_Month'] = test_df['Start Date'].dt.month
test_df['Start_Day'] = test_df['Start Date'].dt.day
test_df['Start_Weekday'] = test_df['Start Date'].dt.weekday  # Monday=0

# From End Date
test_df['End_Year'] = test_df['End Date'].dt.year
test_df['End_Month'] = test_df['End Date'].dt.month
test_df['End_Day'] = test_df['End Date'].dt.day
test_df['End_Weekday'] = test_df['End Date'].dt.weekday


In [29]:
test_df.drop(columns=["Start Date","End Date"],axis=1,inplace=True)

In [30]:
cat_cols=test_df.select_dtypes(include=["object"]).columns

mappings = {}

for col in cat_cols:
    codes, uniques = pd.factorize(test_df[col])
    test_df[col] = codes
    mappings[col] = dict(enumerate(uniques))

# Show mappings for each column
for col, mapping in mappings.items():
    print(f"{col} mapping: {mapping}\n")

Customer Segment 1 mapping: {0: 'I', 1: 'F', 2: 'K', 3: 'H', 4: 'J', 5: 'G', 6: 'L'}

Customer Segment 2 mapping: {0: 'A', 1: 'D', 2: 'C', 3: 'B', 4: 'E'}

Customer Segment 3 mapping: {0: 'AA', 1: 'BB', 2: 'DD', 3: 'FF', 4: 'CC', 5: 'EE', 6: 'GG'}

Customer Segment 4 mapping: {0: 'CCC'}

Customer Segment 5 mapping: {0: 'AB', 1: 'CD'}

Activity 1 mapping: {0: 'EF'}

Activity 2 mapping: {0: 'IJ'}

Country mapping: {0: 'GHI', 1: 'PQR', 2: 'MNO', 3: 'ABC', 4: 'VWX'}

City mapping: {0: 'LNO392', 1: 'FUH372', 2: 'DIF777', 3: 'ISH651', 4: 'QGD703', 5: 'OHG455', 6: 'GHK468', 7: 'IKB920', 8: 'IQH555', 9: 'LJK362'}

Product mapping: {0: 'VUI150', 1: 'IEJ418', 2: 'AHU981', 3: 'ASH122', 4: 'WGG471', 5: 'OXC337', 6: 'SBS339', 7: 'DDH740', 8: 'TWV318', 9: 'VSY183', 10: 'WWQ705', 11: 'IYR218', 12: 'RWN676', 13: 'ELT244', 14: 'RPZ341', 15: 'UKC231', 16: 'ZZZ772', 17: 'TUH861', 18: 'JNQ525', 19: 'EFK842', 20: 'BZD102', 21: 'VTJ209', 22: 'UXG876', 23: 'BNX965', 24: 'YET951', 25: 'OQJ243', 26: 'LUU257', 

In [31]:
test_df.head()

,Estimated Win Rate,Price,Customer Segment 1,Customer Segment 2,Customer Segment 3,Customer Segment 4,Customer Segment 5,Unit Number,Activity 1,Activity 2,Country,City,Product,Competitor,Marketing Source,Division,Start_Year,Start_Month,Start_Day,Start_Weekday,End_Year,End_Month,End_Day,End_Weekday
0,1.00,3000.000000,0,0,0,0,0,2.0,0,0,0,0,0,0,0,0,2018,2,11,6,2018,9,4,1
1,0.75,250000.000000,0,0,0,0,0,1.0,0,0,0,0,1,0,0,1,2018,11,14,2,2019,3,5,1
2,0.00,139677.231907,0,1,0,0,0,1.0,0,0,1,1,2,1,0,2,2024,5,16,3,2024,8,26,0
3,0.00,5194.576510,0,1,0,0,0,1.0,0,0,2,2,3,0,1,0,2023,2,9,3,2023,8,21,0
4,0.00,132065.504490,0,0,0,0,0,1.0,0,0,1,1,2,1,2,2,2025,2,11,1,2025,12,31,2


In [32]:
test_pred_proba = model.predict_proba(test_df)[:, 1]

test_pred_proba = (test_pred_proba >= 0.5).astype(int)
submission = pd.DataFrame({"ID": Id,"TARGET": test_pred_proba})

submission.to_csv("submission.csv", index=False)

In [33]:
df=pd.read_csv("/kaggle/working/submission.csv")

df.head()

,ID,TARGET
0,177087,1
1,177088,0
2,177089,0
3,177090,0
4,177091,0
